# 第5章 收益率与风险度量 —— 配套代码

对应正文 `book/part2/05-returns-risk.md`。依赖内置数据，离线可跑。

**本 notebook 演示内容**：
1. 数据加载与两种收益率对比
2. 年化指标计算
3. 收益分布的矩（均值/方差/偏度/峰度）
4. 下行风险：半方差与下行标准差
5. 索提诺比率、卡尔玛比率实现
6. 风险收益总览表（含全套指标）
7. 累计净值 + 水下曲线双子图
8. VaR：历史模拟法 vs 参数法 vs 蒙特卡洛法
9. ES（期望损失/CVaR）计算
10. 收益直方图标注 VaR/ES
11. 厚尾导致正态法低估 VaR 的对比
12. 习题参考解答

> **提示**：先运行第1个代码格（Cell 1）生成数据，其余格均依赖该格输出。

In [ ]:
# === 在线运行引导（Google Colab）。本地 / Binder 会自动跳过 ===
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess

    if not os.path.isdir("src/fds"):  # 尚未进入仓库
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/albertandking/financial-data-science", "fds_book"],
                       check=True)
        os.chdir("fds_book")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
        print("已就绪：仓库已克隆、fds 已安装、内置数据随仓库一并克隆。")

In [ ]:
# Cell 1：环境初始化与数据加载
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from fds import (
    load_sample_prices, daily_returns, set_chinese_font,
    annualized_return, annualized_volatility, sharpe_ratio, max_drawdown
)

set_chinese_font()

# 加载内置价格数据（约750交易日，4只A股风格资产）
prices = load_sample_prices()
rets = daily_returns(prices)            # 简单收益率
log_rets = daily_returns(prices, log=True)  # 对数收益率

TRADING_DAYS = 252
print(f"价格数据维度：{prices.shape}")
print(f"收益率数据维度：{rets.shape}")
print(f"\n价格数据（末5行）：")
prices.tail()

## 5.2 两种收益率的对比验证

验证：
- 对数收益率的**时间可加性**：多期对数收益之和 = 总对数收益
- 简单收益率**无法直接相加**（需要连乘）
- 日度级别两者**近似相等**

In [ ]:
# Cell 2：两种收益率的性质验证
ticker = 'TECH'
N = 20  # 取前20个交易日

p_sub = prices[ticker].iloc[:N+1]  # 前21个价格点 -> 20个收益率
r_sub = rets[ticker].iloc[:N]
l_sub = log_rets[ticker].iloc[:N]

# 1. 对数收益率时间可加性验证
sum_log = l_sub.sum()
total_log = np.log(p_sub.iloc[-1] / p_sub.iloc[0])
print(f"对数收益率之和：{sum_log:.8f}")
print(f"期间总对数收益：{total_log:.8f}")
print(f"差异：{abs(sum_log - total_log):.2e}  ← 几乎为零，验证时间可加性 ✓")

# 2. 简单收益率不能直接相加
wrong_sum = r_sub.sum()  # 错误做法：直接相加
correct_cum = (1 + r_sub).prod() - 1  # 正确做法：连乘
print(f"\n简单收益率之和（错误）：{wrong_sum:.4f}")
print(f"简单收益率连乘（正确）：{correct_cum:.4f}")
print(f"误差：{abs(wrong_sum - correct_cum):.4f}  ← 存在可观差异，不可直接相加")

# 3. 日度级别：两者近似相等
daily_diff = (rets - log_rets).abs().mean()
print(f"\n日均简单收益率 vs 对数收益率 绝对差异：")
print(daily_diff.round(6))

## 5.3 年化换算

In [ ]:
# Cell 3：年化收益与年化波动率
ann_ret = annualized_return(rets)
ann_vol = annualized_volatility(rets)

print("年化收益率（几何复合）：")
print(ann_ret.round(4).to_string())
print("\n年化波动率（√252 法则）：")
print(ann_vol.round(4).to_string())

# 手动验证 √T 法则：BANK
sigma_daily = rets['BANK'].std(ddof=1)
sigma_annual_manual = sigma_daily * np.sqrt(TRADING_DAYS)
print(f"\n手动验证 BANK 年化波动：{sigma_annual_manual:.4f}")
print(f"fds 函数结果：{ann_vol['BANK']:.4f}")
print(f"差异：{abs(sigma_annual_manual - ann_vol['BANK']):.2e}")

## 5.4 收益分布的矩

In [ ]:
# Cell 4：收益分布的均值、方差、偏度、峰度
moments = pd.DataFrame({
    '日均收益': rets.mean(),
    '日波动率': rets.std(ddof=1),
    '偏度': rets.apply(stats.skew),
    '峰度（Fisher）': rets.apply(stats.kurtosis),   # Fisher定义：正态=0
    '超额峰度': rets.apply(stats.kurtosis),          # Fisher=超额峰度
})
moments.index.name = '资产'
print("收益分布矩统计：")
print(moments.round(4).to_string())
print("\n注：偏度<0（左偏）→ 极端亏损更可能；超额峰度>0（厚尾）→ 正态假设低估VaR")

## 5.5 下行风险指标：半方差、索提诺、卡尔玛

以下指标 `fds` 包未内置，手动实现。

In [ ]:
# Cell 5：自定义下行风险函数
def semivariance(returns, target=0.0):
    """半方差（Semivariance）：仅统计低于 target 的部分"""
    downside = returns[returns < target] - target
    if len(downside) == 0:
        return 0.0
    return float((downside ** 2).mean())

def downside_std(returns, target=0.0, periods=252):
    """年化下行标准差（Downside Deviation），分母用全部T期"""
    semi = np.sqrt(((np.minimum(returns - target, 0)) ** 2).mean())
    return semi * np.sqrt(periods)

def sortino_ratio(returns, risk_free=0.0, target=0.0, periods=252):
    """年化索提诺比率"""
    excess_mean = returns.mean() - risk_free / periods
    dd = np.sqrt(((np.minimum(returns - target, 0)) ** 2).mean())
    if dd == 0:
        return np.nan
    return excess_mean / dd * np.sqrt(periods)

def calmar_ratio(returns, periods=252):
    """卡尔玛比率 = 年化收益 / 最大回撤绝对值"""
    ann_r = annualized_return(returns)
    mdd = abs(max_drawdown(returns))
    if mdd == 0:
        return np.nan
    return ann_r / mdd

# 验证
print("TECH 半方差：", round(semivariance(rets['TECH']), 6))
print("TECH 年化下行标准差：", round(downside_std(rets['TECH']), 4))
print("TECH 索提诺（rf=2%）：", round(sortino_ratio(rets['TECH'], risk_free=0.02), 4))
print("TECH 卡尔玛比率：", round(calmar_ratio(rets['TECH']), 4))

## 5.6 风险收益总览表

六大指标一览：年化收益 / 年化波动 / 夏普 / 索提诺 / 卡尔玛 / 最大回撤

In [ ]:
# Cell 6：风险收益总览表
rf = 0.02  # 年化无风险利率 2%

summary = pd.DataFrame(index=rets.columns)
summary['年化收益'] = annualized_return(rets)
summary['年化波动'] = annualized_volatility(rets)
summary['夏普比率'] = sharpe_ratio(rets, risk_free=rf)
summary['索提诺比率'] = rets.apply(lambda s: sortino_ratio(s, risk_free=rf))
summary['卡尔玛比率'] = rets.apply(calmar_ratio)
summary['最大回撤'] = rets.apply(max_drawdown)
summary['偏度'] = rets.apply(stats.skew)
summary['超额峰度'] = rets.apply(stats.kurtosis)

print("===== 四只A股风格资产：风险收益全套指标 =====")
print(summary.round(3).to_string())
print()
print("按夏普比率排序：", summary['夏普比率'].sort_values(ascending=False).round(3).to_dict())
print("按索提诺比率排序：", summary['索提诺比率'].sort_values(ascending=False).round(3).to_dict())
print("按卡尔玛比率排序：", summary['卡尔玛比率'].sort_values(ascending=False).round(3).to_dict())

## 5.7 累计净值 + 水下曲线（双子图）

In [ ]:
# Cell 7：所有资产累计净值 + TECH水下曲线
nav_all = (1 + rets).cumprod()

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('四只A股风格资产：累计净值与水下曲线', fontsize=14, fontweight='bold')

for ax, ticker in zip(axes.flat, rets.columns):
    nav = nav_all[ticker]
    drawdown = nav / nav.cummax() - 1
    mdd = max_drawdown(rets[ticker])

    ax2 = ax.twinx()
    nav.plot(ax=ax, color='steelblue', linewidth=1.5, label='净值')
    drawdown.plot(ax=ax2, color='crimson', linewidth=1, alpha=0.7)
    ax2.fill_between(drawdown.index, drawdown.values, 0,
                     color='crimson', alpha=0.15)
    ax.set_title(f'{ticker}  |  MDD={mdd:.1%}')
    ax.set_ylabel('净值', color='steelblue')
    ax2.set_ylabel('回撤', color='crimson')
    ax2.set_ylim(-0.7, 0.05)
    ax.tick_params(axis='y', labelcolor='steelblue')
    ax2.tick_params(axis='y', labelcolor='crimson')

plt.tight_layout()
plt.show()

## 5.8 VaR 三种计算方法

以 TECH 为例，展示：
- **历史模拟法**：经验分位数，无分布假设
- **正态参数法**：$\text{VaR} = -(\mu + z_\alpha \sigma)$
- **蒙特卡洛法**：模拟正态分布，取经验分位数

In [ ]:
# Cell 8：三种VaR/ES计算函数
from scipy.stats import norm

def historical_var_es(series, alpha=0.95):
    """历史模拟法 VaR 与 ES"""
    q = np.quantile(series, 1 - alpha)
    var = -q
    tail = series[series <= q]
    es = -tail.mean() if len(tail) > 0 else var
    return var, es

def parametric_var_es(series, alpha=0.95):
    """正态参数法 VaR 与 ES"""
    mu, sigma = series.mean(), series.std(ddof=1)
    z = norm.ppf(1 - alpha)
    var = -(mu + z * sigma)
    # 正态ES解析公式
    es = -mu + sigma * norm.pdf(z) / (1 - alpha)
    return var, es

def mc_var_es(series, alpha=0.95, n_sim=500_000, seed=42):
    """蒙特卡洛法 VaR 与 ES（正态模拟）"""
    rng = np.random.default_rng(seed)
    mu, sigma = series.mean(), series.std(ddof=1)
    simulated = rng.normal(mu, sigma, n_sim)
    q = np.quantile(simulated, 1 - alpha)
    var = -q
    es = -simulated[simulated <= q].mean()
    return var, es

# 对TECH计算三种方法
x = rets['TECH']
results = []
for alpha in [0.95, 0.99]:
    hv, he = historical_var_es(x, alpha)
    pv, pe = parametric_var_es(x, alpha)
    mv, me = mc_var_es(x, alpha)
    results.append({
        '置信水平': f'{int(alpha*100)}%',
        '历史VaR': round(hv, 4), '历史ES': round(he, 4),
        '参数VaR': round(pv, 4), '参数ES': round(pe, 4),
        '蒙特卡洛VaR': round(mv, 4), '蒙特卡洛ES': round(me, 4),
    })

df_var = pd.DataFrame(results).set_index('置信水平')
print("TECH 三种方法 VaR 与 ES 对比：")
print(df_var.to_string())

## 5.9 收益直方图 + VaR/ES 标注

In [ ]:
# Cell 9：TECH日收益分布直方图，标注95%历史VaR与ES
alpha = 0.95
var95_hist, es95_hist = historical_var_es(x, alpha=alpha)
var95_norm, es95_norm = parametric_var_es(x, alpha=alpha)

fig, ax = plt.subplots(figsize=(11, 5))

# 直方图
n, bins, patches = ax.hist(x, bins=60, color='steelblue', alpha=0.6,
                            density=True, label='经验分布')

# 叠加正态拟合曲线
x_range = np.linspace(x.min(), x.max(), 300)
ax.plot(x_range, norm.pdf(x_range, x.mean(), x.std(ddof=1)),
        'k--', lw=1.5, label='正态拟合')

# 标注VaR与ES
ax.axvline(-var95_hist, color='crimson', lw=2,
           label=f'历史VaR95%={var95_hist:.3f}')
ax.axvline(-es95_hist, color='darkred', lw=2, linestyle='--',
           label=f'历史ES95%={es95_hist:.3f}')
ax.axvline(-var95_norm, color='orange', lw=2,
           label=f'参数VaR95%={var95_norm:.3f}')

# 填充左尾
tail_mask = x_range < -var95_hist
ax.fill_between(x_range[tail_mask],
                norm.pdf(x_range[tail_mask], x.mean(), x.std(ddof=1)),
                alpha=0.3, color='crimson', label='VaR左尾区域')

ax.set_title('TECH 日收益分布：VaR 与 ES 可视化')
ax.set_xlabel('日收益率')
ax.set_ylabel('概率密度')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

## 5.10 厚尾导致正态法低估 VaR：全面对比

In [ ]:
# Cell 10：四只资产 95%/99% VaR 历史 vs 参数对比，量化低估幅度
underest_rows = []

for ticker in rets.columns:
    s = rets[ticker]
    for alpha in [0.95, 0.99]:
        hv, _ = historical_var_es(s, alpha)
        pv, _ = parametric_var_es(s, alpha)
        underest = (hv - pv) / hv * 100  # 正态法低估百分比
        underest_rows.append({
            '资产': ticker,
            '置信水平': f'{int(alpha*100)}%',
            '历史VaR': round(hv, 4),
            '参数VaR': round(pv, 4),
            '低估幅度(%)': round(underest, 1),
            '超额峰度': round(float(stats.kurtosis(s)), 2),
        })

df_under = pd.DataFrame(underest_rows)
print("正态参数法对 VaR 的低估（历史法 vs 参数法）：")
print(df_under.to_string(index=False))
print("\n结论：超额峰度越高，参数法低估越严重（99%比95%差距更大）")

## 5.11 厚尾可视化：QQ图对比

In [ ]:
# Cell 11：QQ图展示厚尾（四只资产 vs 正态分布）
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('四只A股风格资产日收益率 QQ 图（vs 正态分布）', fontsize=13, fontweight='bold')

for ax, ticker in zip(axes.flat, rets.columns):
    r = rets[ticker]
    (osm, osr), (slope, intercept, _) = stats.probplot(r, dist='norm')
    ax.scatter(osm, osr, s=6, alpha=0.5, color='steelblue')
    ax.plot(osm, slope * np.array(osm) + intercept,
            color='crimson', lw=1.5, label='正态参考线')
    kurt = round(float(stats.kurtosis(r)), 2)
    ax.set_title(f'{ticker}  |  超额峰度={kurt}')
    ax.set_xlabel('理论分位数')
    ax.set_ylabel('样本分位数')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
print("QQ图两端偏离参考线 → 厚尾 → 正态假设低估极端事件概率")

## 5.12 ES vs VaR 对比：四只资产尾部特征

In [ ]:
# Cell 12：ES/VaR比率，量化各资产尾部"厚重程度"
alpha = 0.95
tail_df = []
for ticker in rets.columns:
    s = rets[ticker]
    hv, he = historical_var_es(s, alpha)
    tail_df.append({
        '资产': ticker,
        f'历史VaR{int(alpha*100)}%': round(hv, 4),
        f'历史ES{int(alpha*100)}%': round(he, 4),
        'ES/VaR比率': round(he / hv, 3),
        '偏度': round(float(stats.skew(s)), 3),
        '超额峰度': round(float(stats.kurtosis(s)), 3),
    })

df_tail = pd.DataFrame(tail_df).set_index('资产')
print("各资产尾部风险特征（ES/VaR比率越高 → 尾部越重）：")
print(df_tail.to_string())
print("\n注：正态分布下 ES95%/VaR95% ≈ 1.26，高于此值说明尾部比正态更重")

## 5.13 风险收益散点图（所有资产一览）

In [ ]:
# Cell 13：风险收益散点图（年化波动 vs 年化收益），气泡大小=夏普比率
fig, ax = plt.subplots(figsize=(9, 6))

sharpe_vals = sharpe_ratio(rets, risk_free=0.02)
vol_vals = annualized_volatility(rets)
ret_vals = annualized_return(rets)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for i, ticker in enumerate(rets.columns):
    size = max(sharpe_vals[ticker] * 400, 100)  # 气泡大小正比于夏普
    ax.scatter(vol_vals[ticker], ret_vals[ticker],
               s=size, color=colors[i], alpha=0.8, label=ticker,
               edgecolors='white', linewidth=1.5, zorder=3)
    ax.annotate(ticker,
                xy=(vol_vals[ticker], ret_vals[ticker]),
                xytext=(8, 5), textcoords='offset points',
                fontsize=11, fontweight='bold', color=colors[i])

ax.set_xlabel('年化波动率')
ax.set_ylabel('年化收益率')
ax.set_title('风险-收益散点图（气泡大小 = 夏普比率）')
ax.axhline(0.02, color='gray', linestyle='--', lw=1, alpha=0.5, label='无风险利率2%')
ax.legend()
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
plt.tight_layout()
plt.show()

## 5.14 习题参考解答

以下代码对应正文第 5.12 节习题，可直接运行验证。

In [ ]:
# === 习题5.1：收益率性质验证 ===
print("=" * 50)
print("习题5.1：收益率性质验证")
print("=" * 50)

for ticker in ['BANK', 'TECH']:
    p_sub = prices[ticker].iloc[:21]
    r_sub = rets[ticker].iloc[:20]
    l_sub = log_rets[ticker].iloc[:20]

    # (a) 对数收益时间可加性
    sum_log = l_sub.sum()
    total_log = np.log(p_sub.iloc[-1] / p_sub.iloc[0])
    print(f"\n{ticker} (a) 对数收益之和={sum_log:.8f}, 总对数收益={total_log:.8f}, 差={abs(sum_log-total_log):.2e}")

    # (b) 简单收益率不可直接相加
    wrong = r_sub.sum()
    correct = (1 + r_sub).prod() - 1
    print(f"{ticker} (b) 简单收益和（错误）={wrong:.4f}, 连乘（正确）={correct:.4f}, 误差={abs(wrong-correct):.4f}")

    # (c) 小变动近似：|r| < 1%的部分
    small = rets[ticker][rets[ticker].abs() < 0.01]
    small_log = log_rets[ticker][rets[ticker].abs() < 0.01]
    diff_small = (small - small_log).abs().mean()
    all_diff = (rets[ticker] - log_rets[ticker]).abs().mean()
    print(f"{ticker} (c) |r|<1%时差异={diff_small:.6f}，全样本差异={all_diff:.6f}")

In [ ]:
# === 习题5.2：全套风险指标对比 ===
print("=" * 50)
print("习题5.2：全套风险指标对比")
print("=" * 50)

rf = 0.02
ex52 = pd.DataFrame(index=rets.columns)
ex52['年化收益'] = annualized_return(rets)
ex52['年化波动'] = annualized_volatility(rets)
ex52['夏普'] = sharpe_ratio(rets, risk_free=rf)
ex52['索提诺'] = rets.apply(lambda s: sortino_ratio(s, risk_free=rf))
ex52['卡尔玛'] = rets.apply(calmar_ratio)
ex52['最大回撤'] = rets.apply(max_drawdown)

print("\n完整指标表：")
print(ex52.round(3).to_string())
for col in ['夏普', '索提诺', '卡尔玛']:
    print(f"\n按{col}排序：", ex52[col].sort_values(ascending=False).round(3).to_dict())

In [ ]:
# === 习题5.3：三种VaR方法对比 ===
print("=" * 50)
print("习题5.3：TECH三种VaR/ES方法对比")
print("=" * 50)

x_tech = rets['TECH']
ex53_rows = []

for alpha in [0.95, 0.99]:
    hv, he = historical_var_es(x_tech, alpha)
    pv, pe = parametric_var_es(x_tech, alpha)
    mv, me = mc_var_es(x_tech, alpha, n_sim=500_000, seed=42)
    ex53_rows.append({
        '置信水平': f'{int(alpha*100)}%',
        '历史VaR': round(hv, 4), '参数VaR': round(pv, 4), '蒙特卡洛VaR': round(mv, 4),
        '历史ES': round(he, 4), '参数ES': round(pe, 4), '蒙特卡洛ES': round(me, 4),
        '参数法低估(%)': round((hv - pv) / hv * 100, 1),
    })

print(pd.DataFrame(ex53_rows).set_index('置信水平').to_string())
print(f"\nTECH 超额峰度={stats.kurtosis(x_tech):.2f}，厚尾导致正态法系统性低估")

In [ ]:
# === 习题5.4：最大回撤深度与持续期 ===
print("=" * 50)
print("习题5.4：LIQUOR最大回撤深度与持续期")
print("=" * 50)

nav_liq = (1 + rets['LIQUOR']).cumprod()
drawdown_liq = nav_liq / nav_liq.cummax() - 1

# 找最大回撤结束点（净值最低点）
end_date = drawdown_liq.idxmin()
# 找起点：最低点之前净值最高处
start_date = nav_liq.loc[:end_date].idxmax()
mdd_val = drawdown_liq[end_date]

# 计算回撤持续期（交易日数）
n_days = len(nav_liq.loc[start_date:end_date]) - 1

print(f"最大回撤：{mdd_val:.1%}")
print(f"回撤起点（前高）：{start_date.date()}")
print(f"回撤终点（最低）：{end_date.date()}")
print(f"持续交易日数：{n_days}")

# 绘图
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
nav_liq.plot(ax=ax1, color='darkorange', lw=1.5, label='LIQUOR净值')
ax1.axvspan(start_date, end_date, alpha=0.2, color='crimson', label='最大回撤区间')
ax1.set_title('LIQUOR 净值曲线与最大回撤区间', fontsize=12)
ax1.set_ylabel('净值')
ax1.legend()

drawdown_liq.plot(ax=ax2, color='crimson', lw=1.5)
ax2.fill_between(drawdown_liq.index, drawdown_liq.values, 0,
                 color='crimson', alpha=0.2)
ax2.axhline(mdd_val, color='black', lw=1, linestyle='--',
            label=f'MDD={mdd_val:.1%}')
ax2.set_ylabel('回撤')
ax2.set_xlabel('日期')
ax2.legend()
plt.tight_layout()
plt.show()

In [ ]:
# === 习题5.5：ES/VaR比率与尾部厚度 ===
print("=" * 50)
print("习题5.5：ES/VaR比率与尾部厚度分析")
print("=" * 50)

alpha = 0.95
ex55 = []
for ticker in rets.columns:
    s = rets[ticker]
    hv, he = historical_var_es(s, alpha)
    ratio = he / hv
    ex55.append({
        '资产': ticker,
        '历史VaR95%': round(hv, 4),
        '历史ES95%': round(he, 4),
        'ES/VaR': round(ratio, 3),
        '偏度': round(float(stats.skew(s)), 3),
        '超额峰度': round(float(stats.kurtosis(s)), 3),
    })

df55 = pd.DataFrame(ex55).set_index('资产').sort_values('ES/VaR', ascending=False)
print(df55.to_string())
print("\n正态分布理论ES/VaR95% ≈", round(1 / (1 - alpha) * norm.pdf(norm.ppf(1-alpha)), 3))
print("各资产均高于正态理论值，证实厚尾特征")
print("\n结论：ES/VaR最高者尾部最厚，一旦突破VaR后损失幅度远超正态预期")